# Solution: Traiter des valeurs manquantes

Les données manquantes font parties du quotidien du data scientist. Rares sont les jeux de données qui n'en ont pas. Concrètement le problème est que la grande majorité des algorithmes d'apprentissage ne savent pas les gérer. Avec `sklearn`, l'erreur qui revient régulièrement est la suivante:

    ValueError: Input contains NaN, infinity or a value too large for dtype('float64')

Cette erreur arrive dès que notre dataframe contient au moins un `None` ou bien un `np.nan` ("nan" vient de l'anglais "not a number"). La gestion de données manquantes fait partie du pré-traitement du jeu de données. On peut la faire indépendemment de l'algorithme d'apprentissage utilisé par la suite.

Heureusement, plusieurs méthodes sont à notre disposition : du très simple au plus compliqué. 

Il faut toujours garder en tête que quoi qu'on fasse on navigue dans l'inconnu et qu'il n'y a pas de remède miracle.

## Remplacer par une statistique

La technique la plus classique consiste à remplacer chaque valeur manquante par une statistique calculée sur la variable concernée. Le plus souvent on utilise la moyenne ou bien la médiane comme valeur de substitution. 

In [1]:
import pandas as pd


df = pd.DataFrame({
    'poids': [4, 1, 3, 100, None],
    'taille': [42, None, None, None, None]
})

* Utilisez la fonctionnalité [pandas.DataFrame.fillna](https://pandas.pydata.org/pandas-docs/version/0.21/generated/pandas.DataFrame.fillna.html) afin de valoriser les données manquantes. Vous opterez pour les deux stratégies de substitution évoquées ci-dessus (moyenne puis valeur médiane)

In [2]:
df['poids_mean'] = df['poids'].fillna(df.poids.mean())
df['taille_mean'] = df['taille'].fillna(df.taille.mean())

df['poids_median'] = df['poids'].fillna(df.poids.median())
df['taille_median'] = df['taille'].fillna(df.taille.median())

df

,poids,taille,poids_mean,taille_mean,poids_median,taille_median
0,4.0,42.0,4.0,42.0,4.0,42.0
1,1.0,NaN,1.0,42.0,1.0,42.0
2,3.0,NaN,3.0,42.0,3.0,42.0
3,100.0,NaN,100.0,42.0,100.0,42.0
4,NaN,NaN,27.0,42.0,3.5,42.0


* La classe `SimpleImputer` du module `impute` du module `sklearn` ([sklearn.impute.SimpleImputer](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html)) permet de régler le même problème de façon similaire.


In [3]:
import pandas as pd


df = pd.DataFrame({
    'poids': [4, 1, 3, 100, None],
    'taille': [42, None, None, None, None]
})
df

,poids,taille
0,4.0,42.0
1,1.0,NaN
2,3.0,NaN
3,100.0,NaN
4,NaN,NaN


* Avec le stratégie 'mean'

In [4]:
from sklearn import impute

imputer = impute.SimpleImputer( strategy='mean')

df[:] = imputer.fit_transform(df)
df

,poids,taille
0,4.0,42.0
1,1.0,42.0
2,3.0,42.0
3,100.0,42.0
4,27.0,42.0


⚠️ Il est à noter que les méthodes de `sklearn` prennent en entrée des `arrays` `numpy` ou bien des `DataFrame` `pandas`. Cependant elles retournent toujours des `arrays`. Pour que notre `df` reste un `DataFrame` on utilise `df[:] = imputer.fit_transform(df)` pour remplacer le contenu de `df`, et pas l'objet en lui-même.

Il y a deux problèmes dans ce que l'on a fait. Premièrement, il semble y avoir un déséquilibre dans la variable `poids`. En effet, 3 des 4 valeurs sont petites (4, 1, et 3) alors que la 4ème est grande (100). Le problème c'est que la moyenne est faussée puisque la valeur 100 contribue "trop". Notre but est de remplacer les valeurs manquantes par des valeurs réalistes. Ici, on aimerait refléter le fait que les valeurs sont plus souvent petites que grandes. Dans ce cas, il vaut mieux utiliser la médiane plutôt que la moyenne.

* Avec le stratégie 'median'

In [5]:
df = pd.DataFrame({
    'poids': [4, 1, 3, 100, None],
    'taille': [42, None, None, None, None]
})

imputer = impute.SimpleImputer(strategy='median')

df[:] = imputer.fit_transform(df)
df

,poids,taille
0,4.0,42.0
1,1.0,42.0
2,3.0,42.0
3,100.0,42.0
4,3.5,42.0


La valeur 3.5 obtenue avec la médiane semble plus réaliste que la valeur 27 obtenue avec la moyenne.

Le second problème concerne la variable `taille`. En effet, elle ne contient qu'une seule valeur, ce qui fait que sa moyenne et sa médiane valent toutes les deux 42. Le problème est que la moyenne est biaisée et ne reflète probablement pas la réalité. Dans ce cas là, il vaut mieux supprimer la variable comme expliqué dans la section suivante.

## Supprimer les lignes concernées

Le plus simple reste tout bonnement de supprimer les observations qui contiennent des valeurs manquantes.

In [6]:
df = pd.DataFrame({
    'couleur': ['bleu', 'rouge', 'bleu', None, None],
    'poids': [None, 42, 24, 242, None],
    'taille': [180, 13, 140, 134, 16]
})

df

,couleur,poids,taille
0,bleu,NaN,180
1,rouge,42.0,13
2,bleu,24.0,140
3,None,242.0,134
4,None,NaN,16


* En utilisant [pandas.DataFrame.dropna](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.dropna.html), supprimez une ligne dès lors qu'elle contient au moins une valeur nulle

In [7]:
df.dropna(axis='rows', how='any')

,couleur,poids,taille
1,rouge,42.0,13
2,bleu,24.0,140


Evidemment cette approche n'est pas conseillée dans le cas où trop de lignes sont supprimées. Cependant, cela vaut largement le coup si vous perdez seulement une petite portion du jeu de données initial. Le seuil acceptable varie au cas par cas. Mais disons qu'on peut largement se passer de 10% d'un jeu de données.

Une alternative à cette approche consiste en la suppression des lignes qui ont au moins un nombre $n$ de valeurs manquantes. 

Intuitivement si une observation a une seule valeur manquante on suppose qu'on ne va pas introduire trop de bruit en la remplacant par la moyenne de la variable correspondante. Cependant plus une observation contient de valeurs manquantes à traiter, plus ce bruit va être important en remplacant ces valeurs et plus on va s'éloigner de la réalité.

* En utilisant [pandas.DataFrame.dropna](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.dropna.html), supprimez une ligne dès lors qu'elle contient un minimum de deux valeurs manquantes

In [8]:
# On supprime toutes les lignes qui ont plus d'une valeur manquante
df.dropna(axis='rows', thresh=1)

,couleur,poids,taille
0,bleu,NaN,180
1,rouge,42.0,13
2,bleu,24.0,140
3,None,242.0,134
4,None,NaN,16


Il se peut aussi que les valeurs manquantes ne concernent que certaines variables (colonnes) du jeu de données. Dans ce cas, il est légitime d'envisager la suppression de la variable en question. 

> Une variable qui a 50% de valeurs manquantes ne va pas servir à grand chose.

* En utilisant [pandas.DataFrame.dropna](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.dropna.html), supprimez une colonne dès lors qu'elle contient au moins une valeurs manquantes

In [9]:
df.dropna(axis='columns', how='any')

,taille
0,180
1,13
2,140
3,134
4,16


## Créer une nouvelle modalité

La fait qu'une valeur soit manquante peut évidemment avoir du sens. Par exemple, quelqu'un qui n'a pas indiqué sa religion sur Facebook est possiblement agnostique ou sans religion. Une méthode simple consiste donc à remplacer les valeurs manquantes d'une variable par une constante. Dans le cas d'une variable numérique, on peut utiliser une valeur aberrante qui n'apparaît pas, par exemple dans le cas suivant on utilise -1. Cependant il vaut mieux remplacer par une valeur statistique comme décrit plus haut et réserver cette méthode pour des variables catégorielles.

In [10]:
df = pd.DataFrame({
    'religion': ['boudhisme', 'catholicisme', None, 'zoroastrianisme', None],
    'poids': [None, 63, 73, 58, None],
})

* Substituez les valeurs manquantes de la colonne `religion` par la valeur "pas indiquée" et les valeurs manquantes de la colonne `poids` par `-1`.


In [11]:
df['religion'].fillna('pas indiquée', inplace=True)
df['poids'].fillna(-1, inplace=True)

df

,religion,poids
0,boudhisme,-1.0
1,catholicisme,63.0
2,pas indiquée,73.0
3,zoroastrianisme,58.0
4,pas indiquée,-1.0


## Compter le nombre de valeurs manquantes

Dans certain cas, on peut  compter le nombre de valeurs manquantes par observation pour en faire une nouvelle variable. Ceci peut avoir beaucoup de sens dans certains cas. Par exemple, un assuré qui n'a pas renseigné beaucoup d'informations est peut-être moins digne de confiance qu'un autre avec un dossier complet.

In [12]:
df = pd.DataFrame({
    'religion': ['boudhisme', 'catholicisme', None, 'zoroastrianisme', None],
    'poids': [None, 63, 73, 58, None],
})
df

,religion,poids
0,boudhisme,NaN
1,catholicisme,63.0
2,None,73.0
3,zoroastrianisme,58.0
4,None,NaN


* Rajoutez la colonne `n_valeurs_manquantes` à df. Cette colonne contiendra  le nombre d'éléments nuls de l'observation. Vous pourrez utiliser les fonctions [pandas.isnull](https://pandas.pydata.org/pandas-docs/version/0.21/generated/pandas.isnull.html) et [pandas.DataFrame.sum](https://pandas.pydata.org/pandas-docs/version/0.21/generated/pandas.DataFrame.sum.html)

In [13]:
df.isnull().sum(axis='index')

religion    2
poids       2
dtype: int64

In [14]:
# TODO : rajoutez la colonne `n_valeurs_manquantes`
df['n_valeurs_manquantes'] = df.isnull().sum(axis='columns')
df

,religion,poids,n_valeurs_manquantes
0,boudhisme,NaN,1
1,catholicisme,63.0,0
2,None,73.0,1
3,zoroastrianisme,58.0,0
4,None,NaN,2


## Cas spécifique des séries temporelles

Reprenons l'exemple précédent où on remplace les valeurs manquantes d'une variable par sa moyenne. Cette facon de faire peut poser problème si on a trop de données manquantes puisque la moyenne ne sera pas assez robuste. C'est d'autant plus le cas pour des données qui ont une dimension temporelle. En effet, si on prend la moyenne globale et qu'on remplace les valeurs manquantes avec, alors il se peut qu'on introduise de l'information du futur dans le présent. Par exemple prenons le cas suivant.

In [15]:
import datetime as dt

df = pd.DataFrame({
    'date': [dt.date(2017, 1, i+1) for i in range(8)],
    'valeur': [12, 13, None, 14, None, 20, 22, None],
})

df['valeur_avec_moyenne'] = df['valeur'].fillna(df['valeur'].mean())

df

,date,valeur,valeur_avec_moyenne
0,2017-01-01,12.0,12.0
1,2017-01-02,13.0,13.0
2,2017-01-03,NaN,16.2
3,2017-01-04,14.0,14.0
4,2017-01-05,NaN,16.2
5,2017-01-06,20.0,20.0
6,2017-01-07,22.0,22.0
7,2017-01-08,NaN,16.2


En machine learning, il est important de ne pas "tricher", c'est à dire d'entraîner sur des informations qu'on n'est pas censé connaître durant la phase d'entraînement. On s'impose cette contrainte parce qu'on veut avoir un modèle généraliste qui fonctionne sur des données qu'il n'a pas vu durant sa phase d'entraînement. Dans une situation réelle, on applique un modèle sur des nouvelles données, pour lesquelles on ne connaît évidemment pas le futur. Il faut inclure cette aspect temporel durant le preprocessing et la phase d'entraînement.

Au lieu de remplacer les valeurs manquantes par la moyenne globale, on va utiliser une moyenne mobile qui prend seulement en compte les valeurs passées. De cette façon notre modèle ne saura rien du futur durant l'apprentissage.

* Utilisez la fonction [pandas.DataFrame.rolling](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.rolling.html) afin de valoriser les valeurs manquantes da la varibale `valeur` par la moyenne mobile sur une fenetre de 5 observations. 

In [16]:
rolling = df.rolling(window=len(df), min_periods=1)
rolling_mean = rolling['valeur'].mean()
df['valeur_avec_moyenne_mobile'] = df['valeur'].fillna(rolling_mean)

df

,date,valeur,valeur_avec_moyenne,valeur_avec_moyenne_mobile
0,2017-01-01,12.0,12.0,12.0
1,2017-01-02,13.0,13.0,13.0
2,2017-01-03,NaN,16.2,12.5
3,2017-01-04,14.0,14.0,14.0
4,2017-01-05,NaN,16.2,13.0
5,2017-01-06,20.0,20.0,20.0
6,2017-01-07,22.0,22.0,22.0
7,2017-01-08,NaN,16.2,16.2


Pour la première valeur manquante, on voit qu'on la remplace par 12.5 (la moyenne des deux premières valeurs) au lieu de 16.2 (la moyenne globale). Ce qui est important c'est qu'on a respecté la dimension temporelle des données et qu'on a pas introduit d'information illégale -- on appelle cela du *leakage* en anglais.